# AI as a Mathematical Collaborator — Weather modelling workshop

**UDSM MATH4SDG · Saturday 21 November 2026 · about 60 minutes**

**Goal:** Use generative AI to construct **two different models** for estimating a missing temperature measurement. Compare their predictions, question the AI's explanations, and make your own evidence-based choice.

We use **wholly synthetic daily observations** at two imaginary nearby weather stations. These are not real Tanzanian weather observations, nor data from TMA or any other partner. The dataset supports no conclusions about actual climate.

You will work in JupyterLab **and** with an AI assistant, individually or in pairs. Record the AI suggestions you received, what you changed, and which claims you checked independently. Work through the notebook in order. **Do not use Run All before writing the model functions.**

**Confidentiality:** Do not paste restricted industrial information, personal records, or partner data into external AI services. The synthetic classroom dataset may be discussed safely.


## 1 · Explore the data (about 10 minutes)

**Scenario:** On some days, the temperature at a *target weather station* is missing. We still have the temperature at a nearby station, plus humidity and wind speed measured on the same day. Can we estimate the target temperature?

Our file `ai_weather_development.csv` contains **270 daily observations**, from **1 January to 27 September 2025**. Each row represents **one day** at two imaginary nearby stations. These are **synthetic classroom measurements**, not real observations from TMA.

| Column | What it tells us |
|---|---|
| `date` | Day of observation |
| `nearby_temp_c` | Temperature at the nearby station (°C) |
| `humidity_pct` | Relative humidity (%) |
| `wind_speed_m_s` | Wind speed (m/s) |
| `target_temp_c` | Temperature at the target station (°C), the value we want to estimate |

On these historical days, the target temperature is known. We can therefore use earlier observations to learn how it relates to the other measurements. When evaluating a model on later days, we give it **only** the other measurements; the known target temperature is used **afterward** to check its predictions.

This is **estimating a missing measurement using other measurements from the same day**, not forecasting tomorrow's temperature.

### Step A — Read the file

Open this notebook from the repository's `notebooks/` folder, keeping `data/exercises/` in its original location. The first code cell imports Python tools, reads the CSV file, and shows its first five rows. A CSV is a simple table stored in a text file.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

file_path = "../data/exercises/ai_weather_development.csv"
development = pd.read_csv(file_path)

print("Observations:", len(development))
display(development.head())

### Step B — Put the days in chronological order

We first tell pandas to interpret the `date` column as dates, rather than ordinary text. Next we sort the rows from the **earliest** day to the **latest**. This matters because we want our model to learn from earlier days and be evaluated on later ones.

In [ ]:
development["date"] = pd.to_datetime(development["date"])
development = development.sort_values("date")

print("Earliest date:", development["date"].min().date())
print("Latest date:", development["date"].max().date())

### Step C — Choose training and validation periods

Now the **first 210 rows** are the earliest 210 days (**1 January–29 July**). We use them for **training**: the model may learn from their known target temperatures. The remaining **60 rows** are later days (**30 July–27 September**). We use them for **validation**: we compare model predictions with the target temperatures, without letting the model learn from those 60 answers.

Why 210 and 60? This is approximately a **78%/22% split** of our 270 development observations, close to the common 80%/20% starting point. It gives a substantial training period and still leaves 60 later days for comparison. **The proportions are not a rule:** using more days for training leaves fewer for validation, and using fewer validation days makes the reported error less reliable. Chronological separation is especially appropriate when testing predictions for a later period.

We have also reserved **60 still later days in a separate final-test file**. Do not open it until you have chosen your model. Repeatedly changing a model after seeing validation errors can make the validation score too optimistic; the final test is for one independent check.

In the next cell, `.iloc[:210]` means *the first 210 rows* and `.iloc[210:]` means *all rows from position 210 onward*. Python counts positions starting at zero.

In [ ]:
train = development.iloc[:210].copy()
validation = development.iloc[210:].copy()

print("Training days:", len(train))
print("Validation days:", len(validation))
print("Training dates:", train["date"].min().date(), "to", train["date"].max().date())
print("Validation dates:", validation["date"].min().date(), "to", validation["date"].max().date())

## 2 · Start with a simple baseline (about 5 minutes)

A baseline is the simplest sensible method to compare with. Our baseline predicts that the missing target temperature is **equal to the nearby-station temperature**.

For a set of predictions, we use the **mean absolute error** (MAE):

$$\mathrm{MAE}=\frac{1}{n}\sum_{i=1}^n |y_i-\widehat y_i|.$$

Smaller is better. Since temperatures are measured in °C, the MAE is also in °C.

**How does the code below relate to MAE?** The next cell defines a function called `score`. **In this notebook, `score` computes the MAE**: it obtains predictions from a chosen model and compares them with the observed target temperatures. For example, `score(baseline_predictor, train, validation)` calculates the **baseline’s MAE on the validation days**. You do not need to understand every line of the function yet.


In [ ]:
# Measurements available for prediction, and the answer we want to estimate.
FEATURES = ["nearby_temp_c", "humidity_pct", "wind_speed_m_s"]
TARGET = "target_temp_c"

def baseline_predictor(train_rows, new_rows):
    # This baseline needs no fitting; 'train_rows' is unused.
    return new_rows["nearby_temp_c"].to_numpy()


def score(predictor, train_rows, evaluation_rows):
    # Compute MAE for a predictor on the evaluation period.
    # The predictor receives *no target temperatures* for evaluation days.
    available_inputs = evaluation_rows[FEATURES].copy()
    predictions = np.asarray(predictor(train_rows, available_inputs), dtype=float)
    if predictions.shape != (len(evaluation_rows),):
        raise ValueError("Return one temperature prediction per evaluation row.")
    if not np.isfinite(predictions).all():
        raise ValueError("Predictions contain missing or infinite values.")
    return mean_absolute_error(evaluation_rows[TARGET], predictions)

baseline_mae = score(baseline_predictor, train, validation)
print(f"Validation MAE — nearby-station baseline: {baseline_mae:.3f} °C")


**Before asking AI:** What are the limitations of the baseline? Must a more complicated model perform better?

## 3 · Ask AI to construct the first model (about 10 minutes)

Open ChatGPT or another AI assistant. Ask it to explain a suitable **simple model** and to write one Python function. You may copy and adapt the prompt below.

> I am using pandas, NumPy and scikit-learn in a Jupyter notebook. I have 210 chronological training observations in a DataFrame named `train`. The columns are `nearby_temp_c`, `humidity_pct`, `wind_speed_m_s`, and `target_temp_c`. The first three are available when the target temperature is missing. Suggest a **simple regression model**, explain why it suits a small training dataset, and provide one Python function named **`first_predict_temperature(train_rows, new_rows)`**. The function must learn using `train_rows` and return one temperature prediction for every row of `new_rows`. `new_rows` contains only the three input columns, not the target. Use the packages already installed and do not use validation or final-test target values to fit the model. Give me one code cell.

Copy the AI-generated **function definition** into the next cell and run it. If there is an error, ask AI for help, but examine the suggested correction yourself. Keep this first model unchanged when making the second one.

**Important:** Never give the prediction function the target temperatures for the observations being evaluated. The notebook's `score` helper takes care of keeping them separate.


In [ ]:
# Paste the AI-generated function here and run this cell.
# It should define: first_predict_temperature(train_rows, new_rows)


**Evaluate the first model.** Run the cell below *after* defining `first_predict_temperature`. The `score` helper calculates MAE on the **validation days**. If you get a `NameError`, go back and run your function-definition cell.


In [ ]:
first_model_mae = score(first_predict_temperature, train, validation)

print(f"Baseline validation MAE: {baseline_mae:.3f} °C")
print(f"First model validation MAE: {first_model_mae:.3f} °C")


**Record your first experiment (edit this Markdown cell):**

- What did you ask AI, and which model did it propose? …
- Did the code run? What did you have to fix? …
- What was the validation MAE compared with the baseline? …

## 4 · Ask AI for a *different* second model (about 10 minutes)

A modelling problem can be approached in several ways. Here are some possibilities; you do **not** need to study all of them today.

| Model family | Main idea |
|---|---|
| Constant offset | Add a learned temperature difference to the nearby-station temperature |
| Linear regression | Fit a linear relationship involving the measurements |
| Support Vector Regression (SVR) | Fit a regularized regression model; nonlinear kernels are possible |
| Decision tree | Split observations using measurement thresholds |
| Random forest | Average predictions from many decision trees |
| k-nearest neighbors | Use measurements from similar training days |

Choose a **different model family** from your first one if you can. Ask your AI assistant for a second approach. A different model is not automatically better.

> My first predictor uses **[NAME YOUR FIRST MODEL]**. Please propose a **substantially different regression model** for the same temperature-prediction problem. Explain how it differs from my first model, and why it might predict better **or worse**. Write a new Python function named **`second_predict_temperature(train_rows, new_rows)`**. Keep the same input/output contract and use only the existing packages. Fit on `train_rows` only; do not use validation or final-test targets for training or model selection. If the algorithm is sensitive to feature scales, use a scikit-learn pipeline with scaling fitted on the training data.

Run the new function-definition cell below, then calculate its validation MAE. **Do not replace or overwrite the first predictor.**


In [ ]:
# Paste the AI-generated second function here and run this cell.
# It should define: second_predict_temperature(train_rows, new_rows)


In [ ]:
second_model_mae = score(second_predict_temperature, train, validation)

print(f"Baseline validation MAE: {baseline_mae:.3f} °C")
print(f"First model validation MAE: {first_model_mae:.3f} °C")
print(f"Second model validation MAE: {second_model_mae:.3f} °C")


## 5 · Ask AI to compare, criticize and improve (about 10 minutes)

**First think for yourselves.** Which model has the smaller validation MAE? Is the difference large enough to be interesting? Does either model improve on the baseline? A lower MAE on one validation period is evidence, not a guarantee of success in other conditions.

Now return to your AI assistant. You can give it the names, mathematical descriptions, and **validation scores** of your two models, then ask:

> We constructed two temperature predictors: **[FIRST MODEL]** and **[SECOND MODEL]**. Their validation MAEs were **[FIRST MAE]** and **[SECOND MAE]** °C; the nearby-station baseline MAE was **[BASELINE MAE]** °C. Compare the mathematical assumptions of these models and explain why their performance might differ. Criticize both approaches: consider overfitting, input scaling, data leakage and changes over time. Suggest **one specific improvement or diagnostic test**, explain why it might help *and* how it might fail. Do not claim an improvement unless we test it.

**Check a claim independently.** For example, verify that a model uses only the allowed input columns, confirm the validation result in JupyterLab, or inspect an unexpected prediction. AI's explanation alone is not evidence.

If time allows, ask AI for a third function named `improved_predict_temperature(train_rows, new_rows)` implementing **one** proposed improvement. Use the optional cells below to evaluate it on the **same validation period**. Otherwise skip those cells and proceed to model selection. **Do not inspect the final test while developing or comparing models.**

**Experiment notes (edit this Markdown cell):**

- Mathematical difference between my first and second models: …
- What did AI criticize about each one? …
- One AI claim I checked independently, and how I checked it: …
- Optional change I tried and its validation MAE: …


In [ ]:
# OPTIONAL: Paste an improved_predict_temperature function here.
# Leave this cell unchanged if you are not trying an improvement.


In [ ]:
# OPTIONAL: After defining the improved function, remove the # signs below.
# improved_model_mae = score(improved_predict_temperature, train, validation)
# print(f"Improved model validation MAE: {improved_model_mae:.3f} °C")


## 6 · Choose a predictor *before* the final test (about 2 minutes)

Compare the validation MAEs of the **baseline**, **first model**, **second model**, and (only if you tried it) the **improved model**. Decide which approach you trust most, using the validation results *and* the mathematical reasoning you recorded. The most complicated model need not win.

In the next code cell, assign **the function itself** to `chosen_predict_temperature` using **one** of these names:

- `baseline_predictor`
- `first_predict_temperature`
- `second_predict_temperature`
- `improved_predict_temperature` (only if you defined it)

For instance, write `chosen_predict_temperature = first_predict_temperature`. **Do not add parentheses**: here we are choosing a function, not calling it. Run the cell and write down your choice and reason before opening the final-test dataset.

**My choice and reason (edit this Markdown cell):** …


In [ ]:
# Replace FUNCTION_NAME with the name of the predictor you choose.
# Remove the # to execute your assignment. Use no parentheses.
# chosen_predict_temperature = FUNCTION_NAME


## 7 · Independent final test (about 5 minutes)

**STOP:** The facilitator will distribute the separate final-test file only after models are chosen. Place `ai_weather_final.csv` in `data/exercises/`, preserving its file name and folder location.

First, select your predictor in the cell immediately above this section. Then change `READY_FOR_FINAL_TEST = False` to `True` in the test cell. The final observations come from a **later time period**. We fit the already-chosen modelling method on **all 270 development days** and evaluate it only once on the 60 withheld days. Never choose a different model based on the final-test result.

The test file includes target temperatures so that we can measure MAE, but our `score` helper sends the model **only input measurements** for final-test days.


In [ ]:
READY_FOR_FINAL_TEST = False  # Change to True only when the final test is released.

if READY_FOR_FINAL_TEST:
    final_path = "../data/exercises/ai_weather_final.csv"
    final_test = pd.read_csv(final_path)

    final_baseline_mae = score(baseline_predictor, development, final_test)
    final_chosen_mae = score(chosen_predict_temperature, development, final_test)
    print(f"Final-test MAE — nearby-station baseline: {final_baseline_mae:.3f} °C")
    print(f"Final-test MAE — chosen model: {final_chosen_mae:.3f} °C")
    print("Lower is better. Report the result even if the baseline wins.")
else:
    print("Final test locked; do not unlock until you have chosen your model.")


## 8 · Compare results across the five groups (final 12 minutes)

**Report to the facilitator:** the model families you tried, the predictor you selected *before* testing, its **validation MAE**, its **final-test MAE**, and one AI suggestion you independently checked. We can build the following table together on the projector or board:

| Group | First model | Second model | Chosen model | Test MAE (°C) | Better than baseline? |
|---|---|---|---|---|---|
| 1 | — | — | — | — | — |
| 2 | — | — | — | — | — |
| 3 | — | — | — | — | — |
| 4 | — | — | — | — | — |
| 5 | — | — | — | — | — |

The **baseline final-test MAE** should be the same for all groups because everyone evaluates the same observations. Compare your test MAE against that common benchmark, not against another group's validation MAE.

**Questions for discussion:** Did the groups choose different mathematical models? Did the more elaborate models perform better? Did AI make accurate predictions about which approach would work best? Did students identify incorrect code or misleading explanations? How could this method of working with AI help your actual industrial investigation?

**Scientific cautions:** The data are wholly synthetic, so these comparisons do not establish what works on real Tanzanian weather records. We assumed humidity, wind and nearby-station temperature remain available when target-station temperature is missing. If missing observations occur especially during extreme conditions, the test results may be optimistic. Different AI prompts, algorithms and random variation are mixed together, so this small workshop cannot show which prompting strategy is generally best.

**Takeaway:** Ask AI to propose competing models. Use mathematical understanding, independent validation and critical discussion to decide what deserves trust.
